### Notebook 5 - Intelligent Traffic Forecasting

Develop a machine learning model capable of predicting daily traffic volume across Brisbane's active transport monitoring network.
This project develops a **universal prediction model** where the transport mode itself is treated as an input feature.

The model will learn from:

- Historical traffic behaviour
- Weather conditions
- Temporal patterns
- Spatial characteristics
- Transport mode

The final model will be capable of estimating expected cyclist, pedestrian or scooter traffic for any monitoring site under different weather and calendar conditions.

This notebook also evaluates multiple machine learning algorithms and explains their predictions using feature importance analysis.

In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)

In [7]:
master_weather = pd.read_parquet(
    "../data/processed/master_weather.parquet"
)

master_weather.head(8)

,Date,Year,Month,Month_Name,Day,Day_of_Week,Weekday_Weekend,Site_ID,Site Name,Mode,Count,Latitude,Longitude,Type,Start_Date,Max_Temp,Min_Temp,Rainfall,Max_Wind,Rainy_Day,Heavy_Rain,Hot_Day,Cold_Morning,Comfortable_Weather,Temp_Range,Windy_Day,Sunshine_Hours,Holiday_Season,Month_Sin,Month_Cos,Lag_1,Lag_7,Lag_30,Rolling_Mean_7,Rolling_Mean_30,Rolling_STD_7,EMA_7,Day_Sin,Day_Cos
0,2022-07-03,2022,7,July,3,Sunday,Weekend,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1246.0,-27.478042,153.00035,Automatic,2013,19.4,10.5,0.0,14.9,0,0,0,1,0,8.9,0,10.000000,0,-0.5,-0.866025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.338837e-01,-0.900969
1,2022-07-04,2022,7,July,4,Monday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1573.0,-27.478042,153.00035,Automatic,2013,13.8,10.7,4.9,14.3,1,0,0,1,0,3.1,0,0.000000,0,-0.5,-0.866025,1246.0,NaN,NaN,1246.000000,1246.000000,NaN,1246.000000,-4.338837e-01,-0.900969
2,2022-07-05,2022,7,July,5,Tuesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,610.0,-27.478042,153.00035,Automatic,2013,13.7,10.2,17.0,10.9,1,1,0,1,0,3.5,0,0.000000,0,-0.5,-0.866025,1573.0,NaN,NaN,1409.500000,1409.500000,231.223917,1327.750000,-9.749279e-01,-0.222521
3,2022-07-06,2022,7,July,6,Wednesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1350.0,-27.478042,153.00035,Automatic,2013,18.1,11.3,6.0,15.0,1,0,0,1,0,6.8,0,4.760978,0,-0.5,-0.866025,610.0,NaN,NaN,1143.000000,1143.000000,489.692761,1148.312500,-7.818315e-01,0.623490
4,2022-07-07,2022,7,July,7,Thursday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2399.0,-27.478042,153.00035,Automatic,2013,21.8,12.7,0.0,27.1,0,0,0,1,1,9.1,1,10.000000,0,-0.5,-0.866025,1350.0,NaN,NaN,1194.750000,1194.750000,413.011198,1198.734375,-2.449294e-16,1.000000
5,2022-07-08,2022,7,July,8,Friday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2333.0,-27.478042,153.00035,Automatic,2013,18.0,9.7,0.0,22.7,0,0,0,1,0,8.3,0,10.015189,0,-0.5,-0.866025,2399.0,NaN,NaN,1435.600000,1435.600000,646.511639,1498.800781,7.818315e-01,0.623490
6,2022-07-09,2022,7,July,9,Saturday,Weekend,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2051.0,-27.478042,153.00035,Automatic,2013,17.6,6.9,0.0,18.9,0,0,0,1,0,10.7,0,10.013894,0,-0.5,-0.866025,2333.0,NaN,NaN,1585.166667,1585.166667,684.545810,1707.350586,9.749279e-01,-0.222521
7,2022-07-10,2022,7,July,10,Sunday,Weekend,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2138.0,-27.478042,153.00035,Automatic,2013,18.7,6.4,0.0,13.0,0,0,0,1,0,12.3,0,10.012414,0,-0.5,-0.866025,2051.0,1246.0,NaN,1651.714286,1651.714286,649.232294,1793.262939,4.338837e-01,-0.900969


In [3]:
master_weather.shape

(57752, 39)

In [5]:
master_weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 57752 entries, 0 to 57751
Data columns (total 39 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 57752 non-null  datetime64[us]
 1   Year                 57752 non-null  int64         
 2   Month                57752 non-null  int64         
 3   Month_Name           57752 non-null  str           
 4   Day                  57752 non-null  int64         
 5   Day_of_Week          57752 non-null  str           
 6   Weekday_Weekend      57752 non-null  str           
 7   Site_ID              57752 non-null  str           
 8   Site Name            57752 non-null  str           
 9   Mode                 57752 non-null  str           
 10  Count                57752 non-null  float64       
 11  Latitude             57752 non-null  float64       
 12  Longitude            57752 non-null  float64       
 13  Type                 57752 non-null  str  

In [6]:
missing = (
    master_weather
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]

Lag_30             2070
Lag_7               483
Rolling_STD_7       138
Lag_1                69
EMA_7                69
Rolling_Mean_30      69
Rolling_Mean_7       69
dtype: int64

In [8]:
model_df = master_weather.copy()

In [9]:
#drop unnecessary columns
model_df = model_df.drop(
    columns=[
        "Date",
        "Month_Name",
        "Day_of_Week",
        "Site Name",
        "Start_Date",
        "Type"
    ]
)

In [10]:
# can use 2022 and 2023 data for training, 2024 for validation and 2025 for testing
# instead of direct 70% train 30% test

In [11]:
model_df.shape

(57752, 33)

In [12]:
lag_features = [
    "Lag_1",
    "Lag_7",
    "Lag_30",
    "Rolling_Mean_7",
    "Rolling_Mean_30",
    "Rolling_STD_7",
    "EMA_7"
]

model_df = model_df.dropna(subset=lag_features)

print(model_df.shape)

(55682, 33)


In [13]:
model_df.isnull().sum().sort_values(ascending=False).head(10)

Year                   0
Comfortable_Weather    0
Day_Sin                0
EMA_7                  0
Rolling_STD_7          0
Rolling_Mean_30        0
Rolling_Mean_7         0
Lag_30                 0
Lag_7                  0
Lag_1                  0
dtype: int64

### Train-validation-test split

In [14]:
train_df = model_df[
    model_df["Year"].isin([2022, 2023])
]

In [15]:
valid_df = model_df[
    model_df["Year"] == 2024
]

In [16]:
test_df = model_df[
    model_df["Year"] == 2025
]

In [17]:
print("Training:", train_df.shape)
print("Validation:", valid_df.shape)
print("Testing:", test_df.shape)

Training: (31471, 33)
Validation: (13102, 33)
Testing: (11109, 33)


In [18]:
TARGET = "Count"

In [19]:
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_valid = valid_df.drop(columns=[TARGET])
y_valid = valid_df[TARGET]

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

In [20]:
categorical_features = [
    "Site_ID",
    "Mode",
    "Weekday_Weekend"
]

In [21]:
numerical_features = [
    col
    for col in X_train.columns
    if col not in categorical_features
]

In [22]:
print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

Categorical Features:
['Site_ID', 'Mode', 'Weekday_Weekend']

Numerical Features:
['Year', 'Month', 'Day', 'Latitude', 'Longitude', 'Max_Temp', 'Min_Temp', 'Rainfall', 'Max_Wind', 'Rainy_Day', 'Heavy_Rain', 'Hot_Day', 'Cold_Morning', 'Comfortable_Weather', 'Temp_Range', 'Windy_Day', 'Sunshine_Hours', 'Holiday_Season', 'Month_Sin', 'Month_Cos', 'Lag_1', 'Lag_7', 'Lag_30', 'Rolling_Mean_7', 'Rolling_Mean_30', 'Rolling_STD_7', 'EMA_7', 'Day_Sin', 'Day_Cos']


### Processing pipeline

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

### Developing baseline models
Before using machine learning, establish simple benchmark models that future models must outperform.

#### Baseline 1 - Yesterday's traffic

In [25]:
baseline_pred = X_test["Lag_1"]

In [26]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

In [27]:
baseline_mae = mean_absolute_error(
    y_test,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_pred
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_pred
)

print(f"MAE : {baseline_mae:.2f}")
print(f"RMSE: {baseline_rmse:.2f}")
print(f"R²  : {baseline_r2:.3f}")

MAE : 231.72
RMSE: 574.77
R²  : 0.868


#### Baseline 2 - 7 day rolling average

In [28]:
baseline7_pred = X_test["Rolling_Mean_7"]

In [29]:
baseline7_mae = mean_absolute_error(
    y_test,
    baseline7_pred
)

baseline7_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline7_pred
    )
)

baseline7_r2 = r2_score(
    y_test,
    baseline7_pred
)

print(f"MAE : {baseline7_mae:.2f}")
print(f"RMSE: {baseline7_rmse:.2f}")
print(f"R²  : {baseline7_r2:.3f}")

MAE : 225.41
RMSE: 527.65
R²  : 0.889


In [30]:
baseline_results = pd.DataFrame({
    "Model": [
        "Yesterday's Traffic",
        "7-Day Rolling Average"
    ],
    "MAE": [
        baseline_mae,
        baseline7_mae
    ],
    "RMSE": [
        baseline_rmse,
        baseline7_rmse
    ],
    "R²": [
        baseline_r2,
        baseline7_r2
    ]
})

baseline_results

,Model,MAE,RMSE,R²
0,Yesterday's Traffic,231.722567,574.767066,0.867803
1,7-Day Rolling Average,225.413680,527.645265,0.888591


In [31]:
fig = px.bar(
    baseline_results,
    x="Model",
    y="RMSE",
    text_auto=".1f",
    color="Model",
    title="Baseline Model Comparison (RMSE)"
)

fig.update_layout(showlegend=False)

fig.show()

### Machine Learning

In [32]:
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

In [33]:
model_results = []

In [34]:
def train_and_evaluate(model, model_name):
    """
    Train a regression model, evaluate it,
    and return predictions and performance metrics.
    """

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    # Train
    pipeline.fit(X_train, y_train)

    # Validation predictions
    valid_pred = pipeline.predict(X_valid)

    # Test predictions
    test_pred = pipeline.predict(X_test)

    # Metrics (Validation)
    valid_mae = mean_absolute_error(y_valid, valid_pred)
    valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
    valid_r2 = r2_score(y_valid, valid_pred)

    # Metrics (Test)
    test_mae = mean_absolute_error(y_test, test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    test_r2 = r2_score(y_test, test_pred)

    model_results.append({
        "Model": model_name,
        "Validation MAE": valid_mae,
        "Validation RMSE": valid_rmse,
        "Validation R²": valid_r2,
        "Test MAE": test_mae,
        "Test RMSE": test_rmse,
        "Test R²": test_r2
    })

    return pipeline, test_pred

#### Model 1 - Linear Regression

In [35]:
lr_model, lr_predictions = train_and_evaluate(
    LinearRegression(),
    "Linear Regression"
)

#### Model 2 - Random Forest

In [36]:
rf_model, rf_predictions = train_and_evaluate(

    RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "Random Forest"

)

#### Model 3 - XGBoost

In [37]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [38]:
from xgboost import XGBRegressor

In [39]:
xgb_model, xgb_predictions = train_and_evaluate(

    XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ),

    "XGBoost"

)

#### Model Leaderboard

In [40]:
leaderboard = (
    pd.DataFrame(model_results)
      .sort_values(
          "Test RMSE"
      )
      .reset_index(drop=True)
)

leaderboard

,Model,Validation MAE,Validation RMSE,Validation R²,Test MAE,Test RMSE,Test R²
0,Linear Regression,156.171057,273.995775,0.916042,197.385103,454.630713,0.917291
1,Random Forest,132.559234,243.945145,0.933449,166.755706,454.858844,0.917208
2,XGBoost,127.528193,235.199237,0.938135,182.750213,517.114074,0.892994


In [42]:
fig = px.bar(

    leaderboard,

    x="Model",

    y="Test RMSE",

    color="Model",

    text_auto=".1f",

    title="Machine Learning Model Comparison"

)

fig.update_layout(
    showlegend=False
)

fig.show()

#### Tune RandomForest (difference with LR is only 0.3)
with Hyperparameter tuning

In [43]:
from sklearn.model_selection import RandomizedSearchCV

In [44]:
param_grid = {

    "model__n_estimators": [200, 300, 500],

    "model__max_depth": [10, 20, 30, None],

    "model__min_samples_split": [2, 5, 10],

    "model__min_samples_leaf": [1, 2, 4],

    "model__max_features": ["sqrt", "log2"]

}

In [45]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [46]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=3)

In [47]:
random_search = RandomizedSearchCV(

    estimator=rf_pipeline,

    param_distributions=param_grid,

    n_iter=20,

    cv=tscv,

    scoring="neg_root_mean_squared_error",

    random_state=42,

    n_jobs=-1,

    verbose=2

)

In [48]:
random_search.fit(
    X_train,
    y_train
)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END model__max_depth=None, model__max_features=log2, model__min_samples_leaf=2, model__min_samples_split=5, model__n_estimators=200; total time=   1.1s
[CV] END model__max_depth=None, model__max_features=log2, model__min_samples_leaf=4, model__min_samples_split=10, model__n_estimators=200; total time=   1.2s
[CV] END model__max_depth=30, model__max_features=log2, model__min_samples_leaf=1, model__min_samples_split=5, model__n_estimators=200; total time=   1.9s
[CV] END model__max_depth=None, model__max_features=log2, model__min_samples_leaf=4, model__min_samples_split=10, model__n_estimators=200; total time=   2.5s
[CV] END model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=2, model__min_samples_split=10, model__n_estimators=200; total time=   1.9s
[CV] END model__max_depth=None, model__max_features=log2, model__min_samples_leaf=2, model__min_samples_split=5, model__n_estimators=200; total time=   3

/Users/aryanbhardwaj78/Desktop/smart-intelligence-project/brisbane-smart-traffic-intelligence/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=2, model__min_samples_split=2, model__n_estimators=300; total time=   3.2s
[CV] END model__max_depth=20, model__max_features=sqrt, model__min_samples_leaf=2, model__min_samples_split=5, model__n_estimators=300; total time=   6.7s
[CV] END model__max_depth=10, model__max_features=sqrt, model__min_samples_leaf=4, model__min_samples_split=2, model__n_estimators=200; total time=   1.3s
[CV] END model__max_depth=None, model__max_features=log2, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=300; total time=   9.2s
[CV] END model__max_depth=10, model__max_features=sqrt, model__min_samples_leaf=4, model__min_samples_split=2, model__n_estimators=200; total time=   3.1s
[CV] END model__max_depth=30, model__max_features=log2, model__min_samples_leaf=4, model__min_samples_split=2, model__n_estimators=300; total time=   2.4s
[CV] END model__max_depth=20, model__max_features=sqrt, model__min

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': [10, 20, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximu

In [49]:
print(random_search.best_params_)

{'model__n_estimators': 300, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': 20}


In [50]:
best_rf = random_search.best_estimator_

In [51]:
best_valid_pred = best_rf.predict(
    X_valid
)

In [52]:
best_test_pred = best_rf.predict(
    X_test
)

In [53]:
best_rf_mae = mean_absolute_error(
    y_test,
    best_test_pred
)

best_rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        best_test_pred
    )
)

best_rf_r2 = r2_score(
    y_test,
    best_test_pred
)

print(f"MAE : {best_rf_mae:.2f}")
print(f"RMSE: {best_rf_rmse:.2f}")
print(f"R²  : {best_rf_r2:.3f}")

MAE : 203.70
RMSE: 609.46
R²  : 0.851


In [54]:
leaderboard = pd.concat(
    [
        leaderboard,
        pd.DataFrame({
            "Model": ["Optimized Random Forest"],
            "Validation MAE": [
                mean_absolute_error(
                    y_valid,
                    best_valid_pred
                )
            ],
            "Validation RMSE": [
                np.sqrt(
                    mean_squared_error(
                        y_valid,
                        best_valid_pred
                    )
                )
            ],
            "Validation R²": [
                r2_score(
                    y_valid,
                    best_valid_pred
                )
            ],
            "Test MAE": [best_rf_mae],
            "Test RMSE": [best_rf_rmse],
            "Test R²": [best_rf_r2]
        })
    ],
    ignore_index=True
)

leaderboard = leaderboard.sort_values(
    "Test RMSE"
)

leaderboard

,Model,Validation MAE,Validation RMSE,Validation R²,Test MAE,Test RMSE,Test R²
0,Linear Regression,156.171057,273.995775,0.916042,197.385103,454.630713,0.917291
1,Random Forest,132.559234,243.945145,0.933449,166.755706,454.858844,0.917208
2,XGBoost,127.528193,235.199237,0.938135,182.750213,517.114074,0.892994
3,Optimized Random Forest,132.164257,239.233112,0.935995,203.703277,609.464018,0.851361


In [55]:
fig = px.bar(

    leaderboard,

    x="Model",

    y="Test RMSE",

    color="Model",

    text_auto=".1f",

    title="Machine Learning Model Comparison"

)

fig.update_layout(
    showlegend=False
)

fig.show()

##### Since Linear regression ahs the best model suitability, we move ahead with it and inspect its coefficients

#### Model Explainability

In [56]:
lr_pipeline = lr_model

In [57]:
feature_names = lr_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

In [58]:
coefficients = lr_pipeline.named_steps[
    "model"
].coef_

In [59]:
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()

coef_df = coef_df.sort_values(
    "Abs_Coefficient",
    ascending=False
)

coef_df.head(20)

,Feature,Coefficient,Abs_Coefficient
26,cat__Site_ID_A025,133.264337,133.264337
6,cat__Site_ID_A005W,97.498561,97.498561
20,cat__Site_ID_A019,88.377585,88.377585
0,cat__Site_ID_A001,69.886814,69.886814
52,remainder__Hot_Day,-57.365907,57.365907
51,remainder__Heavy_Rain,-56.538884,56.538884
12,cat__Site_ID_A011,-38.529770,38.529770
50,remainder__Rainy_Day,-33.754209,33.754209
21,cat__Site_ID_A020,-33.427209,33.427209
18,cat__Site_ID_A017,-31.726100,31.726100


In [60]:
fig = px.bar(

    coef_df.head(20),

    x="Coefficient",

    y="Feature",

    orientation="h",

    color="Coefficient",

    title="Top 20 Most Influential Features (Linear Regression)"

)

fig.show()

### Scenario stimulator

In [64]:
def get_base_scenario(site_id=None, mode=None):
    """
    Returns a real observation that can be modified.
    """

    scenario = X_test.copy()

    if site_id is not None:
        scenario = scenario[
            scenario["Site_ID"] == site_id
        ]

    if mode is not None:
        scenario = scenario[
            scenario["Mode"] == mode
        ]

    scenario = scenario.iloc[[0]].copy()

    return scenario

In [65]:
scenario = get_base_scenario(
    site_id="A001",
    mode="Cyclist"
)

scenario.head()

,Year,Month,Day,Weekday_Weekend,Site_ID,Mode,Latitude,Longitude,Max_Temp,Min_Temp,Rainfall,Max_Wind,Rainy_Day,Heavy_Rain,Hot_Day,Cold_Morning,Comfortable_Weather,Temp_Range,Windy_Day,Sunshine_Hours,Holiday_Season,Month_Sin,Month_Cos,Lag_1,Lag_7,Lag_30,Rolling_Mean_7,Rolling_Mean_30,Rolling_STD_7,EMA_7,Day_Sin,Day_Cos
778,2025,5,15,Weekday,A001,Cyclist,-27.478042,153.00035,23.0,16.2,2.7,9.7,1,0,0,0,0,6.8,0,10.0,0,0.5,-0.866025,2174.0,892.0,1975.0,1547.428571,2328.466667,938.599996,1982.510104,0.781831,0.62349


In [67]:
def update_weather(
    scenario,
    rainfall=None,
    max_temp=None,
    min_temp=None,
    wind=None,
    sunshine=None
):
    """
    Update weather variables in a scenario and automatically
    recalculate dependent engineered features.
    """

    # Rainfall
    if rainfall is not None:
        scenario["Rainfall"] = rainfall
        scenario["Rainy_Day"] = int(rainfall > 0)
        scenario["Heavy_Rain"] = int(rainfall >= 10)

    # Maximum Temperature
    if max_temp is not None:
        scenario["Max_Temp"] = max_temp
        scenario["Hot_Day"] = int(max_temp >= 30)

    # Minimum Temperature
    if min_temp is not None:
        scenario["Min_Temp"] = min_temp
        scenario["Cold_Morning"] = int(min_temp <= 10)

    # Temperature Range
    if (max_temp is not None) or (min_temp is not None):
        scenario["Temp_Range"] = (
            scenario["Max_Temp"] - scenario["Min_Temp"]
        )

        scenario["Comfortable_Weather"] = int(
            (20 <= scenario["Max_Temp"].iloc[0] <= 28) and
            (scenario["Rainfall"].iloc[0] == 0)
        )

    # Wind
    if wind is not None:
        scenario["Max_Wind"] = wind
        scenario["Windy_Day"] = int(wind >= 25)

    # Sunshine
    if sunshine is not None:
        scenario["Sunshine_Hours"] = sunshine
        scenario["Sunshine_Duration"] = sunshine

    return scenario

In [68]:
def predict_scenario(scenario):

    prediction = lr_model.predict(scenario)

    return round(prediction[0],0)

In [69]:
scenario = get_base_scenario(
    site_id="A001",
    mode="Cyclist"
)

scenario = update_weather(

    scenario,

    rainfall=18,

    max_temp=22,

    wind=30,

    sunshine=2

)

predict_scenario(scenario)

np.float64(1421.0)

In [70]:
# -----------------------------
# Traffic Intelligence Engine
# -----------------------------

HOT_DAY_TEMP = 30
HEAVY_RAIN_MM = 10
WINDY_SPEED = 25
COLD_MORNING_TEMP = 10


def simulate_traffic(
    site_id,
    mode,
    rainfall=None,
    max_temp=None,
    min_temp=None,
    wind=None,
    sunshine=None,
    return_scenario=False
):
    """
    Simulates traffic under different weather conditions.

    Parameters
    ----------
    site_id : str
    mode : str
    rainfall : float
    max_temp : float
    min_temp : float
    wind : float
    sunshine : float

    Returns
    -------
    Predicted traffic count
    """

    # 1. Get a real observation
    scenario = get_base_scenario(site_id, mode)

    # 2. Update weather
    scenario = update_weather(
        scenario,
        rainfall=rainfall,
        max_temp=max_temp,
        min_temp=min_temp,
        wind=wind,
        sunshine=sunshine
    )

    # 3. Predict
    prediction = lr_model.predict(scenario)[0]

    print("=" * 55)
    print("🚦 Brisbane Traffic Intelligence System")
    print("=" * 55)
    print(f"Site                : {site_id}")
    print(f"Mode                : {mode}")
    print(f"Predicted Traffic   : {prediction:.0f} movements/day")
    print("=" * 55)

    if return_scenario:
        return prediction, scenario

    return prediction

In [71]:
#example 1
simulate_traffic(
    site_id="A001",
    mode="Cyclist"
)

🚦 Brisbane Traffic Intelligence System
Site                : A001
Mode                : Cyclist
Predicted Traffic   : 1701 movements/day


np.float64(1700.7288381590333)

In [72]:
#example 2
simulate_traffic(
    site_id="A001",
    mode="Cyclist",
    rainfall=18,
    max_temp=24,
    min_temp=17,
    wind=28,
    sunshine=2
)

🚦 Brisbane Traffic Intelligence System
Site                : A001
Mode                : Cyclist
Predicted Traffic   : 1431 movements/day


np.float64(1431.2063731467351)

In [73]:
#example 3
simulate_traffic(
    site_id="A025",
    mode="Pedestrian",
    rainfall=0,
    max_temp=23,
    min_temp=12,
    sunshine=10
)

🚦 Brisbane Traffic Intelligence System
Site                : A025
Mode                : Pedestrian
Predicted Traffic   : 5117 movements/day


np.float64(5117.311869022327)

### Traffic Intelligence System

In [74]:
SCENARIOS = {

    "☀️ Sunny Day": {
        "rainfall": 0,
        "max_temp": 24,
        "min_temp": 14,
        "wind": 10,
        "sunshine": 9
    },

    "🌧️ Heavy Rain": {
        "rainfall": 25,
        "max_temp": 22,
        "min_temp": 18,
        "wind": 20,
        "sunshine": 2
    },

    "🔥 Heatwave": {
        "rainfall": 0,
        "max_temp": 37,
        "min_temp": 27,
        "wind": 12,
        "sunshine": 11
    },

    "🌬️ Windy Day": {
        "rainfall": 0,
        "max_temp": 24,
        "min_temp": 15,
        "wind": 35,
        "sunshine": 7
    },

    "🌤️ Perfect Weekend": {
        "rainfall": 0,
        "max_temp": 23,
        "min_temp": 15,
        "wind": 8,
        "sunshine": 10
    }

}

In [75]:
def compare_weather_scenarios(site_id, mode):

    results = []

    for scenario_name, weather in SCENARIOS.items():

        prediction = simulate_traffic(
            site_id=site_id,
            mode=mode,
            rainfall=weather["rainfall"],
            max_temp=weather["max_temp"],
            min_temp=weather["min_temp"],
            wind=weather["wind"],
            sunshine=weather["sunshine"]
        )

        results.append({
            "Scenario": scenario_name,
            "Predicted Traffic": round(prediction)
        })

    return pd.DataFrame(results)

In [84]:
scenario_results = compare_weather_scenarios(
    site_id="A001",
    mode="Cyclist"
)

scenario_results

🚦 Brisbane Traffic Intelligence System
Site                : A001
Mode                : Cyclist
Predicted Traffic   : 1729 movements/day
🚦 Brisbane Traffic Intelligence System
Site                : A001
Mode                : Cyclist
Predicted Traffic   : 1453 movements/day
🚦 Brisbane Traffic Intelligence System
Site                : A001
Mode                : Cyclist
Predicted Traffic   : 1742 movements/day
🚦 Brisbane Traffic Intelligence System
Site                : A001
Mode                : Cyclist
Predicted Traffic   : 1622 movements/day
🚦 Brisbane Traffic Intelligence System
Site                : A001
Mode                : Cyclist
Predicted Traffic   : 1747 movements/day


,Scenario,Predicted Traffic
0,☀️ Sunny Day,1729
1,🌧️ Heavy Rain,1453
2,🔥 Heatwave,1742
3,🌬️ Windy Day,1622
4,🌤️ Perfect Weekend,1747


In [85]:
fig = px.bar(

    scenario_results,

    x="Scenario",

    y="Predicted Traffic",

    color="Scenario",

    text="Predicted Traffic",

    title="Traffic Intelligence System: Scenario Comparison"

)

fig.update_traces(textposition="outside")

fig.update_layout(
    xaxis_title="Scenario",
    yaxis_title="Predicted Daily Traffic",
    showlegend=False
)

fig.show()

In [86]:
best = scenario_results.loc[
    scenario_results["Predicted Traffic"].idxmax()
]

worst = scenario_results.loc[
    scenario_results["Predicted Traffic"].idxmin()
]

print("="*60)
print("🚦 Brisbane Traffic Intelligence Report")
print("="*60)
print()

print(f"Highest Expected Traffic : {best['Scenario']}")
print(f"Predicted Movements      : {best['Predicted Traffic']:,.0f}")

print()

print(f"Lowest Expected Traffic  : {worst['Scenario']}")
print(f"Predicted Movements      : {worst['Predicted Traffic']:,.0f}")

print()

difference = best["Predicted Traffic"] - worst["Predicted Traffic"]

print(f"Difference               : {difference:,.0f} movements/day")

print("="*60)

🚦 Brisbane Traffic Intelligence Report

Highest Expected Traffic : 🌤️ Perfect Weekend
Predicted Movements      : 1,747

Lowest Expected Traffic  : 🌧️ Heavy Rain
Predicted Movements      : 1,453

Difference               : 294 movements/day


In [87]:
# Baseline prediction (Sunny Day)
baseline = scenario_results.loc[
    scenario_results["Scenario"] == "☀️ Sunny Day",
    "Predicted Traffic"
].iloc[0]

# Percentage change compared to baseline
scenario_results["Change vs Sunny (%)"] = (
    (scenario_results["Predicted Traffic"] - baseline)
    / baseline
    * 100
).round(1)

scenario_results

,Scenario,Predicted Traffic,Change vs Sunny (%)
0,☀️ Sunny Day,1729,0.0
1,🌧️ Heavy Rain,1453,-16.0
2,🔥 Heatwave,1742,0.8
3,🌬️ Windy Day,1622,-6.2
4,🌤️ Perfect Weekend,1747,1.0


In [88]:
fig = px.bar(
    scenario_results,
    x="Scenario",
    y="Change vs Sunny (%)",
    color="Change vs Sunny (%)",
    text="Change vs Sunny (%)",
    color_continuous_scale="RdYlGn",
    title="Impact of Different Weather Scenarios Compared to a Sunny Day"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Scenario",
    yaxis_title="% Change in Traffic",
    coloraxis_colorbar_title="% Change"
)

fig.show()

In [89]:
best = scenario_results.loc[
    scenario_results["Predicted Traffic"].idxmax()
]

worst = scenario_results.loc[
    scenario_results["Predicted Traffic"].idxmin()
]

print("=" * 65)
print("🚦 Brisbane Traffic Intelligence Report")
print("=" * 65)

print(f"\nHighest predicted traffic : {best['Scenario']}")
print(f"Expected movements        : {best['Predicted Traffic']:,.0f}")

print(f"\nLowest predicted traffic  : {worst['Scenario']}")
print(f"Expected movements        : {worst['Predicted Traffic']:,.0f}")

print("\nImpact relative to Sunny Day:")

for _, row in scenario_results.iterrows():

    if row["Change vs Sunny (%)"] > 0:
        direction = "increase"
    elif row["Change vs Sunny (%)"] < 0:
        direction = "decrease"
    else:
        direction = "no change"

    print(
        f"• {row['Scenario']}: "
        f"{abs(row['Change vs Sunny (%)']):.1f}% {direction}"
    )

print("=" * 65)

🚦 Brisbane Traffic Intelligence Report

Highest predicted traffic : 🌤️ Perfect Weekend
Expected movements        : 1,747

Lowest predicted traffic  : 🌧️ Heavy Rain
Expected movements        : 1,453

Impact relative to Sunny Day:
• ☀️ Sunny Day: 0.0% no change
• 🌧️ Heavy Rain: 16.0% decrease
• 🔥 Heatwave: 0.8% increase
• 🌬️ Windy Day: 6.2% decrease
• 🌤️ Perfect Weekend: 1.0% increase


### Custom traffic predictor

In [93]:
def custom_predictor(
    site_id,
    mode,
    rainfall,
    max_temp,
    min_temp,
    wind,
    sunshine
):
    """
    Custom Traffic Prediction Engine
    """

    prediction = simulate_traffic(
        site_id=site_id,
        mode=mode,
        rainfall=rainfall,
        max_temp=max_temp,
        min_temp=min_temp,
        wind=wind,
        sunshine=sunshine
    )

    print("\n")
    print("="*60)
    print("🚦 BRISBANE TRAFFIC INTELLIGENCE SYSTEM")
    print("="*60)

    print(f"Site ID              : {site_id}")
    print(f"Transport Mode       : {mode}")
    print(f"Rainfall             : {rainfall} mm")
    print(f"Maximum Temperature  : {max_temp} °C")
    print(f"Minimum Temperature  : {min_temp} °C")
    print(f"Wind Speed           : {wind} km/h")
    print(f"Sunshine             : {sunshine} hours")

    print("-"*60)

    print(f"Predicted Daily Traffic : {prediction:,.0f} movements")

    print("="*60)

    return prediction

In [94]:
custom_predictor(
    site_id="A001",
    mode="Cyclist",
    rainfall=5,
    max_temp=25,
    min_temp=16,
    wind=12,
    sunshine=8
)

🚦 Brisbane Traffic Intelligence System
Site                : A001
Mode                : Cyclist
Predicted Traffic   : 1665 movements/day


🚦 BRISBANE TRAFFIC INTELLIGENCE SYSTEM
Site ID              : A001
Transport Mode       : Cyclist
Rainfall             : 5 mm
Maximum Temperature  : 25 °C
Minimum Temperature  : 16 °C
Wind Speed           : 12 km/h
Sunshine             : 8 hours
------------------------------------------------------------
Predicted Daily Traffic : 1,665 movements


np.float64(1665.090187902555)

In [95]:
#exmaple 2
custom_predictor(
    "A001",
    "Cyclist",
    rainfall=30,
    max_temp=22,
    min_temp=18,
    wind=28,
    sunshine=2
)

🚦 Brisbane Traffic Intelligence System
Site                : A001
Mode                : Cyclist
Predicted Traffic   : 1401 movements/day


🚦 BRISBANE TRAFFIC INTELLIGENCE SYSTEM
Site ID              : A001
Transport Mode       : Cyclist
Rainfall             : 30 mm
Maximum Temperature  : 22 °C
Minimum Temperature  : 18 °C
Wind Speed           : 28 km/h
Sunshine             : 2 hours
------------------------------------------------------------
Predicted Daily Traffic : 1,401 movements


np.float64(1400.9379475128226)

In [96]:
#example 3
custom_predictor(
    "A001",
    "Cyclist",
    rainfall=0,
    max_temp=24,
    min_temp=15,
    wind=8,
    sunshine=10
)

🚦 Brisbane Traffic Intelligence System
Site                : A001
Mode                : Cyclist
Predicted Traffic   : 1750 movements/day


🚦 BRISBANE TRAFFIC INTELLIGENCE SYSTEM
Site ID              : A001
Transport Mode       : Cyclist
Rainfall             : 0 mm
Maximum Temperature  : 24 °C
Minimum Temperature  : 15 °C
Wind Speed           : 8 km/h
Sunshine             : 10 hours
------------------------------------------------------------
Predicted Daily Traffic : 1,750 movements


np.float64(1749.7266747293543)

In [97]:
#example 4
custom_predictor(
    "A001",
    "Cyclist",
    rainfall=0,
    max_temp=37,
    min_temp=27,
    wind=10,
    sunshine=11
)

🚦 Brisbane Traffic Intelligence System
Site                : A001
Mode                : Cyclist
Predicted Traffic   : 1746 movements/day


🚦 BRISBANE TRAFFIC INTELLIGENCE SYSTEM
Site ID              : A001
Transport Mode       : Cyclist
Rainfall             : 0 mm
Maximum Temperature  : 37 °C
Minimum Temperature  : 27 °C
Wind Speed           : 10 km/h
Sunshine             : 11 hours
------------------------------------------------------------
Predicted Daily Traffic : 1,746 movements


np.float64(1746.4288183626486)

### Interpretation

The Traffic Intelligence System enables users to estimate expected daily traffic movements under different weather conditions.

Unlike traditional forecasting models that simply predict future observations, this simulator provides a "what-if" analysis capability. By modifying environmental conditions such as rainfall, temperature, wind speed, and sunshine hours, transport planners can estimate how active transport demand may change under different scenarios.

This functionality transforms the predictive model into a practical decision-support tool for evaluating potential weather impacts on pedestrian, cyclist, and scooter movements across Brisbane's monitoring network.

# Project Conclusion

## Brisbane Smart Traffic Intelligence System

This project demonstrates a complete end-to-end machine learning workflow for analysing and predicting active transport movements across Brisbane.

### Key Achievements

- Collected and integrated traffic monitoring and weather datasets into a unified analytical dataset.
- Performed extensive exploratory data analysis to identify spatial, temporal and seasonal mobility patterns.
- Engineered domain-specific features including weather indicators, rolling averages, lag variables and cyclical time features.
- Evaluated multiple machine learning models, including Linear Regression, Random Forest and XGBoost.
- Selected Linear Regression as the final model after comparative evaluation, achieving the best generalisation performance on unseen data.
- Developed a Traffic Intelligence Engine capable of estimating daily traffic movements under different weather conditions.
- Built scenario-based simulations to support "what-if" analysis for transport planning and infrastructure management.

### Business Value

The developed system can assist transport planners, local councils and urban analysts by:

- Estimating expected pedestrian, cyclist and scooter movements.
- Understanding the impact of adverse weather on active transport.
- Supporting infrastructure planning and operational decision-making.
- Enabling interactive scenario analysis before implementing transport interventions.

### Future Enhancements

The next stage of this project focuses on deploying the prediction engine as an interactive Streamlit web application, allowing users to:

- Select monitoring sites.
- Choose transport modes.
- Specify weather conditions.
- Instantly predict expected daily traffic movements through an intuitive dashboard.

Overall, this project demonstrates how data engineering, exploratory analysis and machine learning can be combined to create a practical Smart City decision-support system.

In [98]:
import joblib

joblib.dump(
    lr_model,
    "../models/traffic_model.pkl"
)

['../models/traffic_model.pkl']

In [99]:
joblib.dump(
    X_train.columns.tolist(),
    "../models/feature_columns.pkl"
)

['../models/feature_columns.pkl']

In [100]:
master_weather.to_parquet(
    "../data/processed/master_weather.parquet",
    index=False
)